# 🚀 DFCB LLM Validation: Does It Scale to Language Modeling?

**Experiment:** Test whether Data-Free Cognitive Bootstrapping (DFCB) works for real language modeling.

**Setup:**
- Mini-GPT: 10-25M parameter GPT-2 style model
- Dream Phase: 5000 steps on random tokens
- Language Modeling: TinyStories corpus
- **Key Metric:** Perplexity convergence (tokens to target)

**Expected Runtime:** 2-3 hours on GPU (T4)

## 📦 Setup: Install Dependencies & Clone Repo

In [ ]:
# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q transformers datasets torch torchvision torchaudio tqdm matplotlib

# Clone repo
!git clone https://github.com/Joemolnos/ZD-CBE.git
%cd ZD-CBE

# Checkout branch
!git checkout claude/zero-data-cognitive-boost-013nitXYkAKikQjZYPXbRMT4

print("\n✅ Setup complete!")

## 🌙 Phase 1: Dream Phase (Random Tokens)

Self-supervised pretraining on random token sequences (no external data).

In [ ]:
# Dream phase: 5000 steps on random tokens
!python dream_language.py \
    --model_size 10M \
    --num_steps 5000 \
    --batch_size 32 \
    --learning_rate 3e-4 \
    --device cuda \
    --save_path dreamer_gpt.pth \
    --seed 42

print("\n✅ Dream phase complete! Dreamer model saved.")

## 📚 Phase 2: Language Modeling (TinyStories)

Train both Random and Dreamer models on TinyStories corpus.

In [ ]:
# Train RANDOM initialization (baseline)
print("\n🎲 Training RANDOM model...")
!python train_language_model.py \
    --model_size 10M \
    --max_tokens 100000 \
    --eval_interval 1000 \
    --batch_size 32 \
    --learning_rate 3e-4 \
    --device cuda \
    --save_dir checkpoints \
    --model_name random_seed42 \
    --seed 42

print("\n✅ Random model training complete!")

In [ ]:
# Train DREAMER initialization (DFCB)
print("\n🌙 Training DREAMER model...")
!python train_language_model.py \
    --model_size 10M \
    --dreamer_checkpoint dreamer_gpt.pth \
    --max_tokens 100000 \
    --eval_interval 1000 \
    --batch_size 32 \
    --learning_rate 3e-4 \
    --device cuda \
    --save_dir checkpoints \
    --model_name dreamer_seed42 \
    --seed 42

print("\n✅ Dreamer model training complete!")

## 📊 Phase 3: Analysis & Visualization

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load metrics
with open('checkpoints/random_seed42_metrics.json', 'r') as f:
    random_metrics = json.load(f)

with open('checkpoints/dreamer_seed42_metrics.json', 'r') as f:
    dreamer_metrics = json.load(f)

# Extract training curves
random_curve = random_metrics['training_curve']
dreamer_curve = dreamer_metrics['training_curve']

# Plot perplexity convergence
fig, ax = plt.subplots(figsize=(10, 6))

random_tokens = [p['tokens'] for p in random_curve if p['perplexity'] is not None]
random_ppl = [p['perplexity'] for p in random_curve if p['perplexity'] is not None]

dreamer_tokens = [p['tokens'] for p in dreamer_curve if p['perplexity'] is not None]
dreamer_ppl = [p['perplexity'] for p in dreamer_curve if p['perplexity'] is not None]

ax.plot(random_tokens, random_ppl, 'o-', label='Random Init', color='red', linewidth=2)
ax.plot(dreamer_tokens, dreamer_ppl, 's-', label='Dreamer Init (DFCB)', color='blue', linewidth=2)

ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Target: 50 ppl')

ax.set_xlabel('Training Tokens', fontsize=12)
ax.set_ylabel('Validation Perplexity', fontsize=12)
ax.set_title('Perplexity Convergence: Random vs Dreamer', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('perplexity_convergence.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved: perplexity_convergence.png")

## 🎯 Phase 4: Sample Efficiency Analysis

In [ ]:
def find_convergence_tokens(curve, target_ppl=50.0):
    """Find tokens needed to reach target perplexity"""
    for point in curve:
        if point['perplexity'] is not None and point['perplexity'] <= target_ppl:
            return point['tokens']
    return None

# Find convergence points
random_conv = find_convergence_tokens(random_curve, 50.0)
dreamer_conv = find_convergence_tokens(dreamer_curve, 50.0)

print("="*60)
print("📊 SAMPLE EFFICIENCY RESULTS")
print("="*60)

if random_conv:
    print(f"Random Init: {random_conv:,} tokens to 50 perplexity")
else:
    print(f"Random Init: Did NOT converge to 50 perplexity")

if dreamer_conv:
    print(f"Dreamer Init: {dreamer_conv:,} tokens to 50 perplexity")
else:
    print(f"Dreamer Init: Did NOT converge to 50 perplexity")

if random_conv and dreamer_conv:
    speedup = random_conv / dreamer_conv
    print(f"\n🚀 SPEEDUP: {speedup:.2f}x")
    print(f"Dreamer needs {(1 - 1/speedup)*100:.1f}% FEWER tokens!")
    
    if speedup >= 2.0:
        print("\n✅ SUCCESS! DFCB scales to language modeling!")
        print("   → Universal method confirmed!")
        print("   → Top venue publication target: NeurIPS/ICML")
    elif speedup >= 1.3:
        print("\n⚠️ PARTIAL SUCCESS: DFCB shows moderate improvement.")
        print("   → Further investigation needed.")
        print("   → Conference paper target: CoLLAs/AAMAS")
    else:
        print("\n❌ LIMITED: DFCB does not scale significantly.")
        print("   → Method limited to symbolic reasoning.")
        print("   → Honest negative result (still publishable!)")
elif dreamer_conv and not random_conv:
    print("\n✅ STRONG SUCCESS! Dreamer converges, Random does NOT!")
    print("   → DFCB provides qualitative improvement!")
else:
    print("\n❌ Both failed to converge. Need longer training or easier target.")

print("\n" + "="*60)
print(f"Final Perplexities:")
print(f"Random: {random_metrics['final_perplexity']:.2f}")
print(f"Dreamer: {dreamer_metrics['final_perplexity']:.2f}")
print("="*60)

## 🔬 Optional: Additional Seeds for Statistical Validation

In [ ]:
# Run with seeds 0, 1, 2 for statistical validation
for seed in [0, 1, 2]:
    print(f"\n{'='*60}")
    print(f"Running seed {seed}")
    print(f"{'='*60}")
    
    # Dream phase
    !python dream_language.py \
        --model_size 10M \
        --num_steps 5000 \
        --batch_size 32 \
        --save_path dreamer_gpt_seed{seed}.pth \
        --seed {seed}
    
    # Random
    !python train_language_model.py \
        --model_size 10M \
        --max_tokens 100000 \
        --model_name random_seed{seed} \
        --seed {seed}
    
    # Dreamer
    !python train_language_model.py \
        --model_size 10M \
        --dreamer_checkpoint dreamer_gpt_seed{seed}.pth \
        --max_tokens 100000 \
        --model_name dreamer_seed{seed} \
        --seed {seed}

print("\n✅ All seeds complete! Ready for statistical analysis.")

## 📥 Download Results

In [ ]:
# Zip all results
!zip -r dfcb_llm_results.zip checkpoints/ *.png *.pth *.json

# Download
from google.colab import files
files.download('dfcb_llm_results.zip')

print("\n✅ Results ready for download!")

## 📝 Next Steps

Based on results:

**If SUCCESS (2× speedup):**
- Write new paper: "DFCB Scales to Language Modeling"
- Target: NeurIPS/ICML main conference
- Key claim: Universal method, not just symbolic reasoning

**If PARTIAL (1.3-2× speedup):**
- Appendix to current paper
- Target: CoLLAs, AAMAS
- Key claim: Promising but needs larger scale

**If NEGATIVE (<1.3× speedup):**
- Honest limitation in paper
- Still publishable (negative results matter!)
- Key claim: DFCB works for symbolic, not language

**Regardless:** Scientific rigor maintained! 🔬✅